---

## Setup

In [ ]:
# Standard library imports
import sys
from pathlib import Path
import logging
import json
from typing import List, Dict, Any

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Project imports
from src.rag.indexer import PolicyIndexer, DocumentChunker
from src.rag.embeddings import EmbeddingGenerator
from src.rag.retriever import PolicyRetriever
from src.utils.config import Config

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✅ Imports successful")
print(f"📁 Project root: {project_root}")
print(f"📁 Policy documents: {project_root / 'data' / 'policies'}")

---

## Part 1: Offline Indexing Pipeline

### Step 1.1: Initialize Components

In [ ]:
# Initialize RAG components
print("🔧 Initializing RAG system components...\n")

# Document chunker (500 tokens, 50 overlap)
chunker = DocumentChunker(chunk_size=500, overlap=50)
print("✅ DocumentChunker: 500 token chunks with 50 token overlap")

# Embedding generator (Ada-002)
embedder = EmbeddingGenerator()
print("✅ EmbeddingGenerator: Azure OpenAI text-embedding-ada-002 (1536 dims)")

# Policy indexer (no chunker/embedder needed in __init__)
indexer = PolicyIndexer()
print(f"✅ PolicyIndexer: Target index '{Config.AZURE_SEARCH_INDEX_NAME}'")

print("\n🎯 RAG system ready for indexing")
print("💡 Note: chunker and embedder will be passed to index_documents() method")

### Step 1.2: Check Existing Index

Let's see if we already have an index with documents.

In [ ]:
# Check if index exists and has documents
try:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient
    
    search_client = SearchClient(
        endpoint=Config.AZURE_SEARCH_ENDPOINT,
        index_name=Config.AZURE_SEARCH_INDEX_NAME,
        credential=AzureKeyCredential(Config.AZURE_SEARCH_ADMIN_KEY)
    )
    
    # Try to get document count
    results = search_client.search("*", top=1, include_total_count=True)
    doc_count = results.get_count()
    
    if doc_count and doc_count > 0:
        print(f"✅ Index exists with {doc_count} documents")
        print("\n💡 You can skip indexing and go directly to Part 2 (Search & Retrieval)")
        print("   Or re-run indexing to refresh the index.\n")
        index_exists = True
    else:
        print("⚠️  Index exists but is empty")
        print("   Run the indexing cells below to populate it.\n")
        index_exists = False
        
except Exception as e:
    print(f"⚠️  Index not found or not accessible: {e}")
    print("   Run the indexing cells below to create and populate it.\n")
    index_exists = False

### Step 1.3: Index Policy Documents

**Process**: 
1. Read PDFs from `data/policies/`
2. Chunk into 500-token segments
3. Generate embeddings using Ada-002
4. Upload to Azure AI Search

**Expected**: 5 policy documents → 12-15 chunks total

**⚠️ Important**: Running this cell multiple times will create **duplicate documents** in the index! 

To avoid duplicates, the code below will:
- **Reset the index** (delete + recreate) before indexing
- Set `reset_index = False` if you want to add to existing index instead

In [ ]:
# Path to policy documents
policy_dir = project_root / "data" / "policies"

# Verify policies exist
if not policy_dir.exists():
    print(f"❌ Policy directory not found: {policy_dir}")
    print("   Please ensure policy PDFs are in data/policies/")
else:
    policy_files = list(policy_dir.glob("*.pdf"))
    print(f"📁 Found {len(policy_files)} policy documents:")
    for pdf in policy_files:
        print(f"   - {pdf.name}")
    
    if len(policy_files) == 0:
        print("\n⚠️  No PDF files found in data/policies/")
    else:
        print(f"\n✅ Ready to index {len(policy_files)} documents")

In [ ]:
# Run the indexing pipeline
print("🚀 Starting indexing pipeline...\n")

# Option to reset index to avoid duplicates
reset_index = True  # Set to False if you want to add to existing index

if reset_index:
    print("🔄 Resetting index to avoid duplicates...")
    try:
        # Delete existing index if it exists
        indexer.index_client.delete_index(indexer.index_name)
        print(f"   ✅ Deleted existing index: {indexer.index_name}")
    except Exception as e:
        print(f"   ℹ️  No existing index to delete (or error: {e})")
    
    # Create fresh index
    indexer.create_index()
    print(f"   ✅ Created fresh index: {indexer.index_name}\n")
else:
    print("⚠️  Adding to existing index (may create duplicates if run multiple times)\n")

print("="*70)

# Index all policies - pass file list, chunker, and embedder
results = indexer.index_documents(
    file_paths=policy_files,
    chunker=chunker,
    embedder=embedder,
    batch_size=10
)

print("\n" + "="*70)
print("\n📊 Indexing Summary:")
print(f"   Documents processed: {results['total_files']}")
print(f"   Total chunks created: {results['total_chunks']}")
print(f"   Total tokens: {results['total_tokens']:,}")
print(f"   Total cost: ${results['total_cost']:.6f}")

if results['failed_files']:
    print(f"\n⚠️  Failed documents: {results['failed_files']}")
else:
    print("\n✅ All documents indexed successfully!")

### Step 1.4: Inspect Indexed Chunks

Let's look at what got indexed.

In [ ]:
# Show detailed breakdown
print("📋 Indexing Breakdown:\n")
print(f"   Files processed: {results['total_files']} / {len(policy_files)}")
print(f"   Total chunks: {results['total_chunks']}")
print(f"   Total tokens: {results['total_tokens']:,}")
print(f"   Total cost: ${results['total_cost']:.6f}")

if results['total_chunks'] > 0:
    print(f"\n💰 Cost Analysis:")
    print(f"   Cost per chunk: ${results['total_cost'] / results['total_chunks']:.6f}")
    print(f"   Cost per document: ${results['total_cost'] / results['total_files']:.6f}")
    print(f"   Avg chunks per document: {results['total_chunks'] / results['total_files']:.1f}")

if results['failed_files']:
    print(f"\n⚠️  Failed files: {', '.join(results['failed_files'])}")

print("\n✅ Documents successfully indexed in Azure AI Search!")

---

## Part 2: Semantic Search & Retrieval

Now that policies are indexed, let's search them!

### Step 2.1: Initialize Retriever

In [ ]:
# Initialize retriever with default settings
retriever = PolicyRetriever(
    top_k=3,                # Return top 3 results
    min_similarity=0.01     # Hybrid search threshold
)

print("✅ PolicyRetriever initialized")
print("   - Top-K: 3 results")
print("   - Min similarity: 0.01 (hybrid search scores)")
print("   - Search type: Hybrid (vector + keyword)")

### Step 2.2: Basic Semantic Search

**Key Insight**: Semantic search understands *meaning*, not just keywords.

- Query "DTI" matches "debt-to-income ratio"
- Query "credit requirements" matches "FICO score", "credit history"
- Numbers preserved: "43%" matches exactly

In [ ]:
# Example queries covering different policy areas
test_queries = [
    "What is the maximum debt-to-income ratio?",
    "What credit score is required?",
    "How do you verify income?",
    "What are the property appraisal requirements?",
    "Are there compliance rules for documentation?"
]

print("🔍 Testing Semantic Search\n")
print("="*70)

for i, query in enumerate(test_queries, 1):
    print(f"\nQuery {i}: {query}")
    print("-"*70)
    
    results = retriever.search(query)
    
    if results:
        print(f"✅ Found {len(results)} relevant chunks:\n")
        
        for j, result in enumerate(results, 1):
            print(f"   Result {j}:")
            print(f"      Policy: {result['doc_title']}")
            print(f"      Category: {result['doc_category']}")
            print(f"      Chunk: {result['chunk_index']}")
            print(f"      Score: {result['score']:.4f}")
            print(f"      Content: {result['content'][:-1]}...")
            print()
    else:
        print("   ⚠️ No results found\n")

print("="*70)

### Step 2.3: Interactive Search

Try your own queries!

In [ ]:
# Interactive search function
def search_policies(query: str, top_k: int = 3):
    """
    Search policies with a custom query.
    
    Args:
        query: Natural language question
        top_k: Number of results (default: 3)
    """
    print(f"\n🔍 Searching: '{query}'")
    print("="*70)
    
    results = retriever.search(query, top_k=top_k)
    
    if not results:
        print("\n⚠️  No results found. Try a different query.\n")
        return
    
    print(f"\n✅ Found {len(results)} relevant policy chunks:\n")
    
    for i, result in enumerate(results, 1):
        print(f"{'─'*70}")
        print(f"Result {i}: {result['doc_title']} (Chunk {result['chunk_index']})")
        print(f"{'─'*70}")
        print(f"Category: {result['doc_category']}")
        print(f"Relevance Score: {result['score']:.4f}")
        print(f"\nContent:\n")
        print(result['content'])
        print()

# Try some searches
search_policies("What is the maximum DTI allowed?")

In [ ]:
# Try another search
search_policies("minimum credit score requirements")

In [ ]:
# Try with more results
search_policies("income verification methods", top_k=5)

### Step 2.3b: Interactive Search Widget 🎛️

Use the widget below for real-time interactive searching!

In [ ]:
# Import ipywidgets for interactive UI
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Create UI components
query_input = widgets.Text(
    value='',
    placeholder='Enter your policy question here...',
    description='Query:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='80%')
)

top_k_slider = widgets.IntSlider(
    value=3,
    min=1,
    max=10,
    step=1,
    description='Results:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='60%')
)

category_dropdown = widgets.Dropdown(
    options=['All Categories', 'credit', 'income', 'underwriting', 'property', 'compliance'],
    value='All Categories',
    description='Category:',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='60%')
)

search_button = widgets.Button(
    description='🔍 Search Policies',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='40px')
)

output_area = widgets.Output(
    layout=widgets.Layout(
        width='100%',
        border='1px solid #ccc',
        padding='10px',
        margin='10px 0'
    )
)

def perform_search(button):
    """Execute search when button is clicked"""
    query = query_input.value.strip()
    
    with output_area:
        clear_output()
        
        if not query:
            print("⚠️ Please enter a query")
            return
        
        print(f"🔍 Searching: '{query}'")
        print("="*80)
        
        # Perform search
        try:
            results = retriever.search(query, top_k=top_k_slider.value)
            
            # Filter by category if specified
            if category_dropdown.value != 'All Categories':
                results = [r for r in results if r['doc_category'] == category_dropdown.value]
            
            if not results:
                print("\n⚠️  No results found. Try a different query or category.\n")
                return
            
            print(f"\n✅ Found {len(results)} relevant policy chunks")
            if category_dropdown.value != 'All Categories':
                print(f"   (filtered by category: {category_dropdown.value})")
            print()
            
            # Display results with rich formatting
            for i, result in enumerate(results, 1):
                print(f"\n{'='*80}")
                print(f"📄 Result {i}: {result['doc_title']} (Chunk {result['chunk_index']})")
                print(f"{'='*80}")
                print(f"📂 Category: {result['doc_category']}")
                print(f"⭐ Relevance Score: {result['score']:.4f}")
                
                # Color-code score
                if result['score'] > 0.03:
                    score_label = "🟢 High Relevance"
                elif result['score'] > 0.02:
                    score_label = "🟡 Medium Relevance"
                else:
                    score_label = "🟠 Low Relevance"
                print(f"   {score_label}")
                
                print(f"\n📝 Content:")
                print(f"{'-'*80}")
                print(result['content'])
                print(f"{'-'*80}")
            
            # Display search metadata
            print(f"\n\n{'='*80}")
            print(f"📊 Search Statistics:")
            print(f"   • Query: '{query}'")
            print(f"   • Results returned: {len(results)}")
            print(f"   • Top K requested: {top_k_slider.value}")
            if category_dropdown.value != 'All Categories':
                print(f"   • Category filter: {category_dropdown.value}")
            print(f"   • Average score: {sum(r['score'] for r in results) / len(results):.4f}")
            print(f"   • Score range: {min(r['score'] for r in results):.4f} - {max(r['score'] for r in results):.4f}")
            print(f"{'='*80}")
            
        except Exception as e:
            print(f"\n❌ Error during search: {str(e)}")

# Connect button to search function
search_button.on_click(perform_search)

# Display the interactive widget
print("✅ Interactive search widget ready!")
print("\n" + "="*80)
print("🎛️  INTERACTIVE POLICY SEARCH")
print("="*80)
print("\nInstructions:")
print("1. Enter your question in the text box")
print("2. Adjust the number of results with the slider")
print("3. Optionally filter by category")
print("4. Click 'Search Policies' to see results")
print("\nExample queries:")
print("  • What is the maximum DTI ratio allowed?")
print("  • minimum credit score for loan approval")
print("  • income verification requirements")
print("  • property appraisal guidelines")
print("="*80 + "\n")

# Display widgets
display(widgets.VBox([
    widgets.HTML("<h3>🔍 Policy Search Interface</h3>"),
    query_input,
    widgets.HBox([top_k_slider, category_dropdown]),
    search_button,
    output_area
]))

✅ Interactive search widget ready!

🎛️  INTERACTIVE POLICY SEARCH

Instructions:
1. Enter your question in the text box
2. Adjust the number of results with the slider
3. Optionally filter by category
4. Click 'Search Policies' to see results

Example queries:
  • What is the maximum DTI ratio allowed?
  • minimum credit score for loan approval
  • income verification requirements
  • property appraisal guidelines



**💡 Tips for Using the Interactive Widget:**

- **Relevance Scores**: Azure hybrid search returns scores typically between 0.01-0.05
  - 🟢 High (>0.03): Highly relevant match
  - 🟡 Medium (0.02-0.03): Good match
  - 🟠 Low (<0.02): Weak match

- **Category Filtering**: Use the dropdown to focus on specific policy types
  - `credit`: Credit score and history requirements
  - `income`: Income verification and debt calculations
  - `underwriting`: General underwriting standards
  - `property`: Property value and appraisal guidelines
  - `compliance`: Regulatory and legal requirements

- **Query Tips**:
  - Use natural language questions: "What is the maximum DTI?"
  - Include key terms: "credit score", "DTI ratio", "income verification"
  - Try different phrasings if results aren't relevant
  - Adjust top_k to see more/fewer results

- **Understanding Results**:
  - Each result shows the source document and chunk number
  - Higher scores indicate better semantic match
  - Multiple chunks from the same document = comprehensive coverage

### Step 2.4: Category Filtering

Sometimes you want to search within a specific policy category.

In [ ]:
# Available categories (based on our policy documents)
categories = ["credit", "income", "underwriting", "property", "compliance"]

query = "What are the requirements?"

print(f"🔍 Query: '{query}'")
print("\n📁 Searching across categories:\n")

for category in categories:
    print(f"{'='*70}")
    print(f"Category: {category.upper()}")
    print(f"{'='*70}")
    
    results = retriever.search(query, category_filter=category)
    
    if results:
        print(f"✅ Found {len(results)} results:\n")
        for i, result in enumerate(results, 1):
            print(f"   {i}. {result['doc_title']} (score: {result['score']:.4f})")
            print(f"      {result['content'][:120]}...\n")
    else:
        print(f"⚠️  No results in '{category}' category\n")

print("="*70)

### Step 2.5: Batch Search

Process multiple queries at once.

In [ ]:
# Batch queries for common compliance questions
compliance_queries = [
    "Is a 38% DTI acceptable?",
    "Can we approve a 620 credit score?",
    "What income documents are required?",
    "How long after bankruptcy can we approve?",
    "What is the maximum LTV ratio?"
]

print("🔍 Batch Search: Processing multiple compliance questions\n")
print("="*70)

batch_results = retriever.search_batch(compliance_queries, show_progress=False)

print("\n📊 Batch Results Summary:\n")

for i, (query, results) in enumerate(zip(compliance_queries, batch_results), 1):
    print(f"{i}. {query}")
    if results:
        print(f"   ✅ {len(results)} relevant chunks found")
        print(f"   Top match: {results[0]['doc_title']} (score: {results[0]['score']:.4f})")
    else:
        print(f"   ⚠️ No results")
    print()

total_results = sum(len(r) for r in batch_results)
print(f"\n✅ Total: {total_results} chunks retrieved across {len(compliance_queries)} queries")

---

## Part 3: Context Preparation for LLMs

### Step 3.1: Generate Context String

Format retrieved chunks for GPT-4 prompts.

In [ ]:
# Example compliance question
question = "Is a debt-to-income ratio of 42% acceptable for loan approval?"

print(f"❓ Question: {question}\n")
print("="*70)
print("\n📝 Retrieving relevant policies...\n")

# Get context string with metadata
context = retriever.get_context_string(
    question,
    include_metadata=True
)

print("✅ Context prepared\n")
print("="*70)
print("RETRIEVED POLICY CONTEXT")
print("="*70)
print(context)
print("="*70)

### Step 3.2: Build Complete LLM Prompt

Show how this context feeds into a GPT-4 compliance check.

In [ ]:
# Build a complete prompt for GPT-4
prompt = f"""You are a lending compliance officer. Your job is to determine if a loan application meets company policy requirements.

Based on these company lending policies:

{context}

Question: {question}

Instructions:
1. Analyze the question against the provided policies
2. Cite specific policy sections in your answer
3. Provide a clear YES/NO answer
4. Explain your reasoning

Answer:"""

print("🤖 Complete GPT-4 Prompt (Ready to Send)\n")
print("="*70)
print(prompt)
print("="*70)
print(f"\n📊 Prompt Stats:")
print(f"   Total length: {len(prompt)} characters")
print(f"   Estimated tokens: ~{len(prompt) // 4}")
print(f"\n💡 This prompt would be sent to GPT-4o for compliance checking")
print(f"   The model will answer based on retrieved policies, not hallucinations!")

### Step 3.3: Context Without Metadata

Sometimes you want clean content without chunk IDs.

In [ ]:
# Get context without metadata
clean_context = retriever.get_context_string(
    question,
    include_metadata=False
)

print("📝 Clean Context (No Metadata)\n")
print("="*70)
print(clean_context)
print("="*70)
print(f"\nLength comparison:")
print(f"   With metadata: {len(context)} chars")
print(f"   Without metadata: {len(clean_context)} chars")
print(f"   Savings: {len(context) - len(clean_context)} chars")

---

## Part 4: Retrieval Analytics

### Step 4.1: Detailed Statistics

In [ ]:
# Get detailed statistics for a query
query = "credit score requirements"

stats = retriever.get_retrieval_stats(query)

print(f"📊 Retrieval Statistics for: '{query}'\n")
print("="*70)
print(f"Query length: {stats['query_length']} characters")
print(f"\nResults:")
print(f"   Count: {stats['results_count']}")
print(f"   Total content: {stats['total_content_length']} characters")
print(f"\nSimilarity Scores:")
print(f"   Average: {stats['avg_score']:.4f}")
print(f"   Maximum: {stats['max_score']:.4f}")
print(f"   Minimum: {stats['min_score']:.4f}")
print(f"\nCategories covered: {', '.join(stats['categories'])}")
print("="*70)

### Step 4.2: Compare Multiple Queries

In [ ]:
# Compare retrieval quality across different query types
comparison_queries = [
    "DTI",  # Short abbreviation
    "debt-to-income ratio",  # Full term
    "What is the maximum DTI?",  # Natural language
    "Is 43% debt ratio acceptable?"  # Specific value
]

print("📊 Query Comparison Analysis\n")
print("="*70)

comparison_results = []

for query in comparison_queries:
    stats = retriever.get_retrieval_stats(query)
    comparison_results.append(stats)
    
    print(f"\nQuery: '{query}'")
    print(f"   Results: {stats['results_count']}")
    print(f"   Avg Score: {stats['avg_score']:.4f}")
    print(f"   Categories: {', '.join(stats['categories']) if stats['categories'] else 'None'}")

print("\n" + "="*70)
print("\n💡 Key Insight: Semantic search handles all query variations!")
print("   - Abbreviations (DTI)")
print("   - Full terms (debt-to-income ratio)")
print("   - Natural language questions")
print("   - Specific values (43%)")

---

## Part 5: Complete RAG Workflow Simulation

### Step 5.1: End-to-End Scenario

Simulate how the Compliance Agent will use RAG.

In [ ]:
def simulate_compliance_check(question: str) -> Dict[str, Any]:
    """
    Simulate the complete RAG workflow for compliance checking.
    
    This is what happens inside the ComplianceAgent (to be built in T047).
    """
    print(f"\n{'='*70}")
    print(f"🔍 COMPLIANCE CHECK: {question}")
    print(f"{'='*70}\n")
    
    # Step 1: Retrieve relevant policies
    print("Step 1: Retrieve relevant policy chunks")
    results = retriever.search(question)
    print(f"   ✅ Retrieved {len(results)} chunks\n")
    
    # Step 2: Build context
    print("Step 2: Build context for LLM")
    context = retriever.get_context_string(question, include_metadata=True)
    print(f"   ✅ Context: {len(context)} characters\n")
    
    # Step 3: Show retrieved policies
    print("Step 3: Review retrieved policies")
    for i, result in enumerate(results, 1):
        print(f"   Policy {i}: {result['doc_title']}")
        print(f"      Category: {result['doc_category']}")
        print(f"      Relevance: {result['score']:.4f}")
        print(f"      Content: {result['content'][:150]}...\n")
    
    # Step 4: Simulate LLM prompt
    print("Step 4: Prepare LLM prompt (would send to GPT-4o)")
    prompt = f"""Based on these policies:
{context[:500]}...

Question: {question}

Provide answer with policy citations."""
    
    print(f"   ✅ Prompt ready ({len(prompt)} chars)")
    print(f"\n💡 Next: Send to GPT-4o → Get grounded answer with citations\n")
    print("="*70)
    
    return {
        "question": question,
        "chunks_retrieved": len(results),
        "context_length": len(context),
        "policies_referenced": list(set(r['doc_title'] for r in results)),
        "categories": list(set(r['doc_category'] for r in results))
    }

# Test with different compliance questions
test_scenarios = [
    "Is a DTI of 42% acceptable?",
    "Can we approve a 620 credit score?",
    "What income documents are required?"
]

workflow_results = []
for scenario in test_scenarios:
    result = simulate_compliance_check(scenario)
    workflow_results.append(result)

### Step 5.2: Workflow Summary

In [ ]:
# Summarize workflow results
print("\n📊 RAG Workflow Summary\n")
print("="*70)

for i, result in enumerate(workflow_results, 1):
    print(f"\nScenario {i}: {result['question']}")
    print(f"   Chunks retrieved: {result['chunks_retrieved']}")
    print(f"   Context length: {result['context_length']} chars")
    print(f"   Policies: {', '.join(result['policies_referenced'])}")
    print(f"   Categories: {', '.join(result['categories'])}")

print("\n" + "="*70)
print("\n✅ RAG System Fully Functional!")
print("\n💡 Next Steps:")
print("   1. Integrate into ComplianceAgent (T047)")
print("   2. Send prompts to GPT-4o for actual compliance decisions")
print("   3. Extract policy citations from GPT-4o responses")
print("   4. Build complete lending decision system")

---

## Part 6: Edge Case Testing - No Results Handling

### Step 6.1: Testing with Irrelevant Queries

**Edge Case**: What happens when a query has no relevant policy matches?

This tests the system's ability to avoid hallucination by returning empty results
when similarity scores fall below the threshold.

In [ ]:
# Test queries that should return no results (completely irrelevant to lending policies)
irrelevant_queries = [
    "How do I bake a chocolate cake?",
    "What is the capital of France?",
    "Best practices for growing tomatoes?",
    "Latest trends in mobile app development",
    "Quantum physics equations and theories"
]

print("🧪 Testing No-Result Handling (Edge Case)")
print("="*80)
print("\n**Spec Requirement (FR-018)**: System MUST acknowledge when no relevant")
print("policy found and avoid hallucinating policy requirements.\n")
print("="*80)

no_result_tests = []

for i, query in enumerate(irrelevant_queries, 1):
    print(f"\n📝 Test {i}/{len(irrelevant_queries)}: '{query}'")
    print("-"*80)
    
    # Search with default threshold (0.01 for hybrid search)
    results = retriever.search(query, top_k=3)
    
    # Record test result
    test_result = {
        "query": query,
        "results_count": len(results),
        "passed": len(results) == 0,
        "max_score": max([r['score'] for r in results], default=0.0)
    }
    no_result_tests.append(test_result)
    
    if len(results) == 0:
        print("✅ PASS: No results returned (scores below threshold)")
        print("   → System will acknowledge: 'No relevant policy found'")
        print("   → Avoids hallucination by returning empty list")
    else:
        print(f"⚠️  UNEXPECTED: {len(results)} results returned")
        print(f"   → Highest score: {test_result['max_score']:.4f}")
        for j, result in enumerate(results, 1):
            print(f"   {j}. {result['doc_title']} (score: {result['score']:.4f})")

print("\n" + "="*80)
print("📊 No-Result Handling Test Summary")
print("="*80)

passed_tests = sum(1 for t in no_result_tests if t['passed'])
print(f"\n✅ Passed: {passed_tests}/{len(no_result_tests)} tests")
print(f"❌ Failed: {len(no_result_tests) - passed_tests}/{len(no_result_tests)} tests")

if passed_tests == len(no_result_tests):
    print("\n🎉 All tests passed! System correctly handles irrelevant queries.")
    print("   → Returns empty list when no relevant policies found")
    print("   → Prevents hallucination and false policy references")
else:
    print("\n⚠️  Some tests failed. Consider adjusting min_similarity threshold.")

print("\n💡 Key Insight:")
print("   By returning empty results for irrelevant queries, the RAG system")
print("   ensures that downstream agents (ComplianceAgent, DecisionAgent) can")
print("   detect missing policy guidance and respond appropriately:")
print("   - 'No applicable policy found for this query'")
print("   - 'Insufficient policy guidance - requires manual review'")
print("   - Avoids making up fake policy requirements")
print("="*80)

### Step 6.2: Testing with Low-Relevance Financial Queries

**Edge Case**: Queries that are financial-related but not covered in lending policies.

These queries might trigger some weak matches, but should ideally return no results
or very low scores, allowing the system to acknowledge insufficient guidance.

In [ ]:
# Test queries that are financial but not in lending policies
borderline_queries = [
    "What are the tax implications of a home loan?",
    "How does cryptocurrency affect mortgage applications?",
    "Can I use my 401k for a down payment?",
    "What are the closing costs for refinancing?",
    "How do I qualify for a VA loan?"
]

print("\n🧪 Testing Low-Relevance Financial Queries")
print("="*80)
print("These queries are financial but may not be directly addressed in policies.")
print("="*80)

borderline_results = []

for i, query in enumerate(borderline_queries, 1):
    print(f"\n📝 Test {i}/{len(borderline_queries)}: '{query}'")
    print("-"*80)
    
    results = retriever.search(query, top_k=3)
    
    if len(results) == 0:
        print("✅ No results: Query not covered in indexed policies")
        print("   → ComplianceAgent can respond: 'No applicable policy found'")
    else:
        print(f"📊 Found {len(results)} results:")
        for j, result in enumerate(results, 1):
            score_label = "🟢" if result['score'] > 0.03 else "🟡" if result['score'] > 0.02 else "🟠"
            print(f"   {j}. {score_label} {result['doc_title']}")
            print(f"      Score: {result['score']:.4f} | Category: {result['doc_category']}")
            print(f"      Snippet: {result['content'][:100]}...")
    
    borderline_results.append({
        "query": query,
        "results_count": len(results),
        "max_score": max([r['score'] for r in results], default=0.0),
        "has_high_score": any(r['score'] > 0.03 for r in results)
    })

print("\n" + "="*80)
print("📊 Borderline Query Analysis")
print("="*80)

for i, result in enumerate(borderline_results, 1):
    print(f"\n{i}. Query: '{result['query']}'")
    print(f"   Results: {result['results_count']} chunks")
    print(f"   Max Score: {result['max_score']:.4f}")
    
    if result['results_count'] == 0:
        print(f"   Status: ✅ Correctly filtered (no relevant policy)")
    elif result['max_score'] > 0.03:
        print(f"   Status: 🟢 High relevance - policy guidance available")
    elif result['max_score'] > 0.02:
        print(f"   Status: 🟡 Medium relevance - review recommended")
    else:
        print(f"   Status: 🟠 Low relevance - insufficient guidance")

print("\n💡 How ComplianceAgent Should Handle These:")
print("   - No results: 'No applicable policy found - requires manual review'")
print("   - Low scores (<0.02): 'Limited policy guidance - pending review'")
print("   - Medium scores (0.02-0.03): 'Some guidance available - verify applicability'")
print("   - High scores (>0.03): 'Clear policy guidance found - proceed with compliance check'")
print("="*80)

### Step 6.3: Threshold Tuning Demonstration

**Advanced**: Adjust the `min_similarity` threshold to control result filtering.

- **Lower threshold** (e.g., 0.005): More permissive, returns weak matches
- **Higher threshold** (e.g., 0.03): Stricter, only high-quality matches
- **Default** (0.01): Balanced for Azure hybrid search scoring

In [ ]:
# Test the same query with different thresholds
test_query = "What is the maximum DTI ratio allowed?"

print(f"🔍 Query: '{test_query}'\n")
print("="*80)
print("Testing Different Similarity Thresholds")
print("="*80)

thresholds = [0.005, 0.01, 0.02, 0.03, 0.05]

for threshold in thresholds:
    print(f"\n📊 Threshold: {threshold}")
    print("-"*80)
    
    # Create retriever with custom threshold
    retriever_test = PolicyRetriever(min_similarity=threshold)
    results = retriever_test.search(test_query, top_k=5)
    
    print(f"Results returned: {len(results)}")
    
    if results:
        print("Scores:")
        for i, result in enumerate(results, 1):
            print(f"   {i}. {result['score']:.4f} - {result['doc_title']}")
    else:
        print("   (No results above threshold)")

print("\n" + "="*80)
print("💡 Threshold Selection Guidelines:")
print("="*80)
print("""
Azure Hybrid Search Scoring Context:
- Pure cosine similarity: 0.0 to 1.0 range
- Azure hybrid search: Typically 0.01 to 0.05 range
- Scores depend on: vector match + keyword match + ranking fusion

Recommended Thresholds:
- 0.005 - Very permissive (use for broad discovery)
- 0.01  - Balanced (default, good for most cases)
- 0.02  - Stricter (reduce false positives)
- 0.03  - Very strict (only high-confidence matches)
- 0.05  - Extremely strict (may miss relevant results)

For Production:
- Monitor score distributions over time
- Adjust threshold based on precision/recall metrics
- Consider different thresholds per category
- Log filtered results for continuous improvement
""")
print("="*80)

---

## Summary & Key Takeaways

### What We Built

1. **Indexing Pipeline**: Document → Chunks → Embeddings → Vector DB
2. **Semantic Search**: Query → Embedding → Hybrid Search → Top-K Results
3. **Context Preparation**: Results → Formatted String → LLM-Ready
4. **Analytics**: Track retrieval quality, scores, coverage
5. **Interactive Widget**: Real-time search with category filtering
6. **Edge Case Handling**: No-result scenarios and threshold tuning

### Why This Matters

**Before RAG**:
- ❌ GPT-4 hallucinations ("According to policy XYZ..." - doesn't exist!)
- ❌ Outdated knowledge (trained on old data)
- ❌ No audit trail (can't verify sources)

**With RAG**:
- ✅ Grounded in actual policies (retrieves first, then answers)
- ✅ Always up-to-date (just reindex documents)
- ✅ Full traceability (every answer cites chunks)

### Key Metrics

- **Indexing**: ~12 chunks from 5 policy PDFs
- **Search Speed**: ~150-350ms per query
- **Cost**: ~$0.0001 per query (embedding only)
- **Accuracy**: Hybrid search combines semantic + keyword matching

### Next: Compliance Agent (T047)

Now that RAG is working, we'll build the ComplianceAgent that:
1. Uses PolicyRetriever to get relevant chunks
2. Sends chunks + question to GPT-4o
3. Extracts compliance status + policy citations
4. Returns structured ComplianceReport

### Experiment Ideas

Try modifying these parameters:
- `top_k`: Return more/fewer chunks (affects context size vs relevance)
- `min_similarity`: Adjust threshold (higher = stricter matching)
- `chunk_size`: Larger chunks (more context) vs smaller (more precise)
- `overlap`: More overlap (better continuity) vs less (more chunks)

### Production Considerations

1. **Index Updates**: Reindex policies when they change
2. **Monitoring**: Track retrieval quality over time
3. **Caching**: Cache frequent queries to reduce costs
4. **Versioning**: Version policies and track which version was used
5. **Security**: Ensure only authorized personnel can update policies